# Rotary position embeddings

This notebook explores pairwise rotations, relative-position behavior, and the repository's `RotaryEmbedding` implementation.

In [1]:
import math
import sys
from pathlib import Path

import torch

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))

torch.manual_seed(7)

## Rotate one vector

A two-dimensional rotation changes direction while preserving length.

In [2]:
x = torch.tensor([1.0, 0.0])
angle = torch.tensor(math.pi / 4)
rotation = torch.tensor([[angle.cos(), -angle.sin()], [angle.sin(), angle.cos()]])
rotated = rotation @ x

print("rotated:", rotated)
print("lengths:", x.norm().item(), rotated.norm().item())

rotated: tensor([0.7071, 0.7071])
lengths: 1.0 1.0


## Use the RoPE implementation

The layer expects `[batch, heads, sequence, dimension]` and rotates adjacent feature pairs.

In [6]:
from src.model.rope import RotaryEmbedding

rope = RotaryEmbedding(dim=32, max_seq_len=64)
x = torch.randn(1, 1, 8, 32)
y = rope(x)

print("input:", x.shape)
print("output:", y.shape)
torch.testing.assert_close(x.square().sum(-1), y.square().sum(-1), atol=1e-5, rtol=1e-5)

input: torch.Size([1, 1, 8, 32])
output: torch.Size([1, 1, 8, 32])


## Check the relative-position property

A common shift should not change the dot product when the relative displacement stays the same.

In [4]:
q = torch.randn(1, 1, 1, 32)
k = torch.randn(1, 1, 1, 32)
score_one = (rope(q, start_pos=8) * rope(k, start_pos=13)).sum()
score_two = (rope(q, start_pos=21) * rope(k, start_pos=26)).sum()

torch.testing.assert_close(score_one, score_two, atol=1e-5, rtol=1e-5)
print("relative-position check passed")

relative-position check passed


RoPE adds fixed positional behavior to queries and keys. It contributes no trainable parameters.